Generating CLIP embeddings for text. Due to CLIP's 77 token limit, we will chunk large transcripts and average the embedding over chunks to create row-wise embeddings

In [14]:
# Imports
import torch
import pandas as pd
import numpy as np
from transformers import CLIPTokenizer, CLIPTextModel
import re
import math

In [15]:
# Using ViT-B-32 CLIP model from OpenCLIP, identical to the one used in our drawing embedding generator
import open_clip
from transformers import CLIPTokenizer

print("Loading CLIP (ViT-B-32)...")

device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device)
model.eval()

# HuggingFace tokenizer used only for chunking (has .encode/.decode API needed by chunk_evenly_balanced)
hf_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

Loading CLIP (ViT-B-32)...


/Scratch/nsahrling/miniconda3/lib/python3.13/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [16]:
# This function removes "Um" and "Uh", but currently the utteranace_masked_filtered is already filtered for this

def remove_fillers(text):
    # Pattern matches standalone "um" or "uh" or "okay" (case insensitive), with optional trailing punctuation
    pattern = r'\b(um+|umm+|uh+|uhh+|okay)\b[\.,;:!?"]*'

    # Replace with a single space, then strip extra whitespace
    cleaned = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # Normalize spacing (e.g., multiple spaces → single space)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    return cleaned


In [17]:
def chunk_evenly_balanced(text, tokenizer, max_length=75):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    max_content_tokens = max_length - tokenizer.num_special_tokens_to_add(pair=False)

    if len(tokens) <= max_content_tokens:
        return [tokenizer.decode(tokens, skip_special_tokens=True)]

    # Split into evenly-sized chunks so each chunk contributes equally when averaged
    num_chunks = math.ceil(len(tokens) / max_content_tokens)
    chunk_size = math.ceil(len(tokens) / num_chunks)

    chunks = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = min(start + chunk_size, len(tokens))
        chunks.append(tokenizer.decode(tokens[start:end], skip_special_tokens=True))

    return chunks

Parallel to our drawing pipeline, we are z-scoring embeddings column wise so that no single dimension (out of 512 possible dimensions) overweights another in downstream analyses

In [18]:
def column_z_score(embedding_series):
    """Z-score a series of embedding arrays column-wise across all rows."""
    valid_mask = embedding_series.notna().values
    arr = embedding_series.values.copy()

    matrix = np.vstack(arr[valid_mask])
    matrix = (matrix - matrix.mean(axis=0)) / matrix.std(axis=0)

    for i, pos in enumerate(np.where(valid_mask)[0]):
        arr[pos] = matrix[i]
    return pd.Series(arr, index=embedding_series.index)

In [19]:
def get_clip_text_embedding(text, clip_tokenizer, model, device, hf_tokenizer):
    if pd.isna(text) or str(text).strip() == "":
        return np.nan

    cleaned_text = remove_fillers(text)
    chunks = chunk_evenly_balanced(cleaned_text, hf_tokenizer)

    chunk_embeddings = []
    with torch.no_grad():
        for chunk in chunks:
            tokens = clip_tokenizer([chunk]).to(device)
            features = model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)
            chunk_embeddings.append(features.cpu().numpy()[0])

    avg = np.mean(chunk_embeddings, axis=0)
    avg = avg / np.linalg.norm(avg)
    return avg

In [20]:
print("Loading CSV...")
utterance_df = pd.read_csv("../data/clean/transcripts_raw_clean.csv")

SEA_ANIMAL_CATEGORIES = ["crab", "octopus", "shark", "whale", "turtle", "seahorse"]

utterance_df = utterance_df[
    utterance_df["target_category"].isin(SEA_ANIMAL_CATEGORIES)
].copy()

Loading CSV...


In [21]:
utterance_df.head()

,participant_id,target_category,utterance_interference,utterance_exists,utterance_ns,utterance_masked,utterance_half_masked_filtered
0,AD001,crab,0,1,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual..."
1,AD001,whale,0,1,Um it looks like a big creature and it has a l...,Um it looks like a big creature and it has a l...,it looks like a big creature and it has a lot ...
2,AD001,turtle,0,1,It has a hard shell and four legs which helps ...,It has a hard shell and four legs which helps ...,It has a hard shell and four legs which helps ...
3,AD001,shark,0,1,It's a big sea creature and it's usually grey....,It's a big sea creature and it's usually grey....,It's a big sea creature and it's usually grey....
4,AD001,seahorse,0,1,Seahorses looks small and it's usually like a ...,It looks small and it's usually like a bright ...,It looks small and it's usually like a bright ...


In [22]:
results = []

for idx, row in utterance_df.iterrows():

    full_utterance = row["utterance_masked"]
    half_utterance = row["utterance_half_masked_filtered"]

    full_embedding = get_clip_text_embedding(
        full_utterance,
        tokenizer,
        model,
        device,
        hf_tokenizer
    )

    half_embedding = get_clip_text_embedding(
        half_utterance,
        tokenizer,
        model,
        device,
        hf_tokenizer
    )

    results.append({
        "participant_id": row["participant_id"],
        "target_category": row["target_category"],
        "utterance_masked": row["utterance_masked"],
        "utterance_half_masked_filtered": row["utterance_half_masked_filtered"],
        "utterance_full_embedding": full_embedding,
        "utterance_half_embedding": half_embedding
    })

output_df = pd.DataFrame(results)
output_df.head()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (83 > 77). Running this sequence through the model will result in indexing errors


,participant_id,target_category,utterance_masked,utterance_half_masked_filtered,utterance_full_embedding,utterance_half_embedding
0,AD001,crab,"It has two claws and two legs, and it is usual...","It has two claws and two legs, and it is usual...","[0.002537407, 0.0024788834, 0.0045457887, -0.0...","[-0.005825282, 0.005044784, -0.012088424, -0.0..."
1,AD001,whale,Um it looks like a big creature and it has a l...,it looks like a big creature and it has a lot ...,"[0.04102609, 0.010294129, -0.0042194705, -0.00...","[0.029467221, 0.021346, -0.01039887, -0.003552..."
2,AD001,turtle,It has a hard shell and four legs which helps ...,It has a hard shell and four legs which helps ...,"[-0.0010793665, 0.033917744, 0.0021302907, -0....","[0.013847926, 0.017997814, -0.0011440668, -0.0..."
3,AD001,shark,It's a big sea creature and it's usually grey....,It's a big sea creature and it's usually grey....,"[0.016167846, 0.004727181, -0.0013396494, -0.0...","[0.018943423, -0.00014422662, -0.0062696417, -..."
4,AD001,seahorse,It looks small and it's usually like a bright ...,It looks small and it's usually like a bright ...,"[0.019036053, 0.010091584, 0.004102062, -0.002...","[0.005921952, -0.007829243, 0.0015856226, -0.0..."


In [23]:
#merge age data
ages = pd.read_csv("../data/clean/participant_age_table.csv")

In [24]:
merged_df = output_df.merge(
    ages[["participant_id", "age_months"]],
    on="participant_id",
    how="left"
)

Just like in drawings, we z-score over columns so no single dimension overweights another in downstram analyses.

In [25]:
# Column-wise z-scoring — consistent with drawing embedding normalization
merged_df["utterance_full_embedding"] = column_z_score(merged_df["utterance_full_embedding"])
merged_df["utterance_half_embedding"] = column_z_score(merged_df["utterance_half_embedding"])

In [26]:
valid = output_df["utterance_full_embedding"].dropna().iloc[0]
print("Embedding dim:", valid.shape)

Embedding dim: (512,)


In [27]:
# Sanity check: after column z-scoring, each column should have mean ≈ 0 and std ≈ 1
def check_z_score(embedding_series, label):
    valid = embedding_series.dropna()
    matrix = np.vstack(valid.values)
    print(f"\n{label}")
    print(f"  Column means — min: {matrix.mean(axis=0).min():.4f}, max: {matrix.mean(axis=0).max():.4f} (expect ≈ 0)")
    print(f"  Column stds  — min: {matrix.std(axis=0).min():.4f},  max: {matrix.std(axis=0).max():.4f}  (expect ≈ 1)")

check_z_score(merged_df["utterance_full_embedding"], "Full utterance embeddings")
check_z_score(merged_df["utterance_half_embedding"], "Half utterance embeddings")


Full utterance embeddings
  Column means — min: -0.0000, max: 0.0000 (expect ≈ 0)
  Column stds  — min: 1.0000,  max: 1.0000  (expect ≈ 1)

Half utterance embeddings
  Column means — min: -0.0000, max: 0.0000 (expect ≈ 0)
  Column stds  — min: 1.0000,  max: 1.0000  (expect ≈ 1)


In [28]:
merged_df.to_csv("../data/embeddings/utterance_half_and_full_embeddings.csv", index=False)